# TabPFN testing

Ran during closed beta testing so code is not easily executable by anyone.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load data
TRAIN_DATA_PATH = Path("data/train_data.parquet")
TEST_DATA_PATH = Path("data/test_data.parquet")
train_df = pd.read_parquet(TRAIN_DATA_PATH)
test_df = pd.read_parquet(TEST_DATA_PATH)

# Convert indices to numeric values
train_df.index = pd.to_numeric(train_df.index)
test_df.index = pd.to_numeric(test_df.index)

# Drop datetime features
cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)

In [ ]:
# Split data into features and targets
train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"].astype(int)
train_full_y_reg = train_df["target_annual_roi"]

test_full_X = test_df.drop(["target", "target_annual_roi"], axis=1)
test_full_y_cat = test_df["target"].astype(int)
test_full_y_reg = test_df["target_annual_roi"]

In [ ]:
# Create subsets of the training data for different sizes
train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_50k_X = train_full_X.tail(50000)
train_50k_y_cat = train_full_y_cat.tail(50000)
train_50k_y_reg = train_full_y_reg.tail(50000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

In [ ]:
# Set API token for TabPFN
import tabpfn_client

tabpfn_client.set_access_token("API_Token_Goes_Here")

In [ ]:
import time
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    mean_squared_error,
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
)
from tabpfn_client import TabPFNClassifier, TabPFNRegressor
from tabpfn_client.constants import ModelVersion


# Function to evaluate TabPFN on a given dataset and save results
def evaluate_tabpfn(
    train_X,
    train_y,
    test_X,
    test_y,
    dataset_size_name,
    task="classification",
    version="v3",
    batch_limit=200000,
):
    # Set output directories based on task type
    if task == "classification":
        out_dir_scores_path = Path("test_results/Classification/scores")
        out_dir_preds_path = Path("test_results/Classification/predictions")
    else:
        out_dir_scores_path = Path("test_results/Regression/scores")
        out_dir_preds_path = Path("test_results/Regression/predictions")

    print(f"\n[{task.upper()}] Evaluating TabPFN on dataset: {dataset_size_name}")

    # Select models and set batch size limits based on version and task type
    if version == "v3":
        max_batch_size = min(200000 - len(train_X), batch_limit)
        if task == "classification":
            model = TabPFNClassifier.create_default_for_version(
                ModelVersion.V3,
                enhanced_fit_mode=True,
                enhanced_fit_mode_metric="roc_auc",
                enhanced_fit_mode_time_limit_s=2400,
                random_state=42,
            )
        else:
            model = TabPFNRegressor.create_default_for_version(
                ModelVersion.V3,
                enhanced_fit_mode=True,
                enhanced_fit_mode_metric="rmse",
                enhanced_fit_mode_time_limit_s=2400,
                random_state=42,
            )
    else:
        max_batch_size = min(50000 - len(train_X), batch_limit)
        if task == "classification":
            model = TabPFNClassifier.create_default_for_version(
                ModelVersion.V2_5, random_state=42
            )
        else:
            model = TabPFNRegressor.create_default_for_version(
                ModelVersion.V2_5, random_state=42
            )

    # Fit the model and measure time
    start_time_fit = time.time()
    model.fit(train_X, train_y)
    fit_time = time.time() - start_time_fit

    # Predict on test data in batches due to limits and measure time
    start_time_pred = time.time()
    test_preds = []
    for i in range(0, len(test_X), max_batch_size):
        batch = test_X.iloc[i : i + max_batch_size]
        try:
            if task == "classification":
                b_preds = model.predict_proba(batch)[:, 1]
            else:
                b_preds = model.predict(batch)
            b_preds = pd.Series(b_preds, index=batch.index).values
            test_preds.append(b_preds)
        # Do not crash if prediction is interrupted, but save what has been processed so far
        except Exception as e:
            print(f"\n[WARNING] Interrupted during Test data prediction.")
            print(f"Exact error: {e}")
            print("Saving partial results.")
            break

    if not test_preds:
        print("No test data batches were processed. Exiting.")
        return

    # Concatenate batch predictions and align with true labels
    test_preds = np.concatenate(test_preds)
    test_pred_time = time.time() - start_time_pred
    test_X = test_X.iloc[: len(test_preds)]
    test_y = test_y[: len(test_preds)]

    # Predict on train data in batches due to limits and measure time
    start_time_pred_train = time.time()
    train_preds = []
    for i in range(0, len(train_X), max_batch_size):
        batch = train_X.iloc[i : i + max_batch_size]
        try:
            if task == "classification":
                b_preds = model.predict_proba(batch)[:, 1]
            else:
                b_preds = model.predict(batch)
            b_preds = pd.Series(b_preds, index=batch.index).values
            train_preds.append(b_preds)
        # Do not crash if prediction is interrupted, but save what has been processed so far
        except Exception as e:
            print(f"\n[WARNING] Interrupted during Train data prediction.")
            print(f"Exact error: {e}")
            print("Saving partial results.")
            break

    if not train_preds:
        print("No train data batches were processed. Exiting.")
        return

    # Concatenate batch predictions and align with true labels
    train_preds = np.concatenate(train_preds)
    train_pred_time = time.time() - start_time_pred_train
    train_X = train_X.iloc[: len(train_preds)]
    train_y = train_y[: len(train_preds)]

    # Create DataFrames for predictions and true labels
    test_preds_df = pd.DataFrame(index=test_X.index)
    test_preds_df["true_y"] = test_y
    test_preds_df["TabPFN"] = test_preds

    train_preds_df = pd.DataFrame(index=train_X.index)
    train_preds_df["true_y"] = train_y
    train_preds_df["TabPFN"] = train_preds

    # Calculate metrics based on task type
    if task == "classification":
        test_roc_auc = roc_auc_score(test_y, test_preds)
        test_pr_auc = average_precision_score(test_y, test_preds)
        train_roc_auc = roc_auc_score(train_y, train_preds)
        train_pr_auc = average_precision_score(train_y, train_preds)

        print(f"Fit Time: {fit_time:.2f} s | Pred Time (Test): {test_pred_time:.2f} s")
        print(f"TEST  - ROC AUC: {test_roc_auc:.4f} | PR AUC: {test_pr_auc:.4f}")
        print(f"TRAIN - ROC AUC: {train_roc_auc:.4f} | PR AUC: {train_pr_auc:.4f}")

        df_scores_test = pd.DataFrame(
            [
                {
                    "Dataset": dataset_size_name,
                    "Model": "TabPFN",
                    "ROC AUC": test_roc_auc,
                    "PR AUC": test_pr_auc,
                    "Fit Time (s)": fit_time,
                    "Pred Time (s)": test_pred_time,
                }
            ]
        )

        df_scores_train = pd.DataFrame(
            [
                {
                    "Dataset": dataset_size_name,
                    "Model": "TabPFN",
                    "ROC AUC": train_roc_auc,
                    "PR AUC": train_pr_auc,
                    "Fit Time (s)": fit_time,
                    "Pred Time (s)": train_pred_time,
                }
            ]
        )

    else:
        test_rmse = np.sqrt(mean_squared_error(test_y, test_preds))
        test_r2 = r2_score(test_y, test_preds)
        test_mae = mean_absolute_error(test_y, test_preds)
        test_mape = mean_absolute_percentage_error(test_y, test_preds)

        train_rmse = np.sqrt(mean_squared_error(train_y, train_preds))
        train_r2 = r2_score(train_y, train_preds)
        train_mae = mean_absolute_error(train_y, train_preds)
        train_mape = mean_absolute_percentage_error(train_y, train_preds)

        print(f"Fit Time: {fit_time:.2f} s | Pred Time (Test): {test_pred_time:.2f} s")
        print(
            f"TEST  - RMSE: {test_rmse:.4f} | R2: {test_r2:.4f} | MAE: {test_mae:.4f} | MAPE: {test_mape:.4f}"
        )
        print(
            f"TRAIN - RMSE: {train_rmse:.4f} | R2: {train_r2:.4f} | MAE: {train_mae:.4f} | MAPE: {train_mape:.4f}"
        )

        df_scores_test = pd.DataFrame(
            [
                {
                    "Dataset": dataset_size_name,
                    "Model": "TabPFN",
                    "RMSE": test_rmse,
                    "R2": test_r2,
                    "MAE": test_mae,
                    "MAPE": test_mape,
                    "Fit Time (s)": fit_time,
                    "Pred Time (s)": test_pred_time,
                }
            ]
        )

        df_scores_train = pd.DataFrame(
            [
                {
                    "Dataset": dataset_size_name,
                    "Model": "TabPFN",
                    "RMSE": train_rmse,
                    "R2": train_r2,
                    "MAE": train_mae,
                    "MAPE": train_mape,
                    "Fit Time (s)": fit_time,
                    "Pred Time (s)": train_pred_time,
                }
            ]
        )

    # Save predictions and scores to files
    test_preds_path = (
        out_dir_preds_path / f"test_preds_{dataset_size_name}_tabpfn.parquet"
    )
    train_preds_path = (
        out_dir_preds_path / f"train_preds_{dataset_size_name}_tabpfn.parquet"
    )
    test_scores_path = (
        out_dir_scores_path / f"test_scores_{dataset_size_name}_tabpfn.csv"
    )
    train_scores_path = (
        out_dir_scores_path / f"train_scores_{dataset_size_name}_tabpfn.csv"
    )

    test_preds_df.to_parquet(test_preds_path)
    train_preds_df.to_parquet(train_preds_path)
    df_scores_test.to_csv(test_scores_path, index=False)
    df_scores_train.to_csv(train_scores_path, index=False)

    print(f"Results saved to {out_dir_scores_path}/ and {out_dir_preds_path}/")

In [ ]:
evaluate_tabpfn(
    train_1k_X,
    train_1k_y_cat,
    test_full_X,
    test_full_y_cat,
    "1k",
    task="classification",
    version="v3",
    batch_limit=200000,
)


[CLASSIFICATION] Evaluating TabPFN on dataset: 1k
00:23 Fitting... Done!
00:26 Predicting... Done!
00:13 Predicting... Done!
00:04 Predicting... Done!
Fit Time: 23.57 s | Pred Time (Test): 40.75 s
TEST  - ROC AUC: 0.6984 | PR AUC: 0.3741
TRAIN - ROC AUC: 1.0000 | PR AUC: 1.0000
Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/


In [ ]:
evaluate_tabpfn(
    train_10k_X,
    train_10k_y_cat,
    test_full_X,
    test_full_y_cat,
    "10k",
    task="classification",
    version="v3",
    batch_limit=200000,
)


[CLASSIFICATION] Evaluating TabPFN on dataset: 10k
09:29 Fitting... Done!
04:17 Predicting... Done!
03:23 Predicting... Done!
00:39 Predicting... Done!
Fit Time: 569.96 s | Pred Time (Test): 461.65 s
TEST  - ROC AUC: 0.7177 | PR AUC: 0.3991
TRAIN - ROC AUC: 0.9915 | PR AUC: 0.9752
Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/


In [ ]:
evaluate_tabpfn(
    train_100k_X,
    train_100k_y_cat,
    test_full_X,
    test_full_y_cat,
    "100k",
    task="classification",
    version="v3",
    batch_limit=30000,
)


[CLASSIFICATION] Evaluating TabPFN on dataset: 100k
00:00 Fitting... \

The provided train set hashes match previously uploaded train sets.


00:00 Fitting... Done!
00:00 Predicting... \

The provided test set hash matches a previously uploaded test set.


04:47 Predicting... Done!
00:00 Predicting... -

...

04:41 Predicting... Done!
00:00 Predicting... -

The provided test set hash matches a previously uploaded test set.


04:36 Predicting... Done!
00:00 Predicting... -

The provided test set hash matches a previously uploaded test set.


04:40 Predicting... Done!
04:42 Predicting... Done!
04:42 Predicting... Done!
04:04 Predicting... Done!
Fit Time: 0.92 s | Pred Time (Test): 2553.61 s
TEST  - ROC AUC: 0.7302 | PR AUC: 0.4175
TRAIN - ROC AUC: 0.8673 | PR AUC: 0.6937
Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/


In [ ]:
evaluate_tabpfn(
    train_1k_X,
    train_1k_y_reg,
    test_full_X,
    test_full_y_reg,
    "1k",
    task="regression",
    version="v3",
    batch_limit=200000,
)


[REGRESSION] Evaluating TabPFN on dataset: 1k
00:26 Fitting... Done!
00:32 Predicting... Done!
00:15 Predicting... Done!
00:04 Predicting... Done!
Fit Time: 26.38 s | Pred Time (Test): 48.40 s
TEST  - RMSE: 0.3609 | R2: 0.0195 | MAE: 0.2388 | MAPE: 60830379826.1013
TRAIN - RMSE: 0.1794 | R2: 0.6838 | MAE: 0.1229 | MAPE: 191520040813.2186
Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/


In [ ]:
evaluate_tabpfn(
    train_10k_X,
    train_10k_y_reg,
    test_full_X,
    test_full_y_reg,
    "10k",
    task="regression",
    version="v3",
    batch_limit=200000,
)


[REGRESSION] Evaluating TabPFN on dataset: 10k
08:41 Fitting... Done!
02:30 Predicting... Done!
01:17 Predicting... Done!
00:32 Predicting... Done!
Fit Time: 521.45 s | Pred Time (Test): 229.27 s
TEST  - RMSE: 0.3589 | R2: 0.0305 | MAE: 0.2397 | MAPE: 49832541019.2019
TRAIN - RMSE: 0.2833 | R2: 0.0962 | MAE: 0.1982 | MAPE: 13876460522.9054
Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/


In [ ]:
evaluate_tabpfn(
    train_50k_X,
    train_50k_y_reg,
    test_full_X,
    test_full_y_reg,
    "50k",
    task="regression",
    version="v3",
    batch_limit=20000,
)


[REGRESSION] Evaluating TabPFN on dataset: 50k
00:00 Fitting... -

The provided train set hashes match previously uploaded train sets.


00:00 Fitting... Done!
04:54 Predicting... Done!
04:55 Predicting... Done!
05:00 Predicting... Done!
04:57 Predicting... Done!
04:54 Predicting... Done!
04:54 Predicting... Done!
15:00 Predicting... /

Exception occurred during _predict, retrying in 0.0 seconds... attempt 1: The read operation timed out


19:51 Predicting... Done!
04:51 Predicting... Done!
04:52 Predicting... Done!
04:58 Predicting... Done!
15:00 Predicting... -

Exception occurred during _predict, retrying in 0.0 seconds... attempt 1: The read operation timed out


30:00 Predicting... |

Giving up _predict(...) after 2 tries (httpx.ReadTimeout: The read operation timed out)
_Predict method failed after 2 attempts. Giving up. Exception: The read operation timed out



[WARNING] Interrupted during Test data prediction.
Exact error: The read operation timed out
Saving partial results.
04:57 Predicting... Done!
04:56 Predicting... Done!
04:26 Predicting... Done!
Fit Time: 0.83 s | Pred Time (Test): 5655.50 s
TEST  - RMSE: 0.3437 | R2: 0.0504 | MAE: 0.2383 | MAPE: 44027433276.4612
TRAIN - RMSE: 0.2889 | R2: 0.1294 | MAE: 0.1944 | MAPE: 11720714932.9458
Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/


In [ ]:
# Completing the remaining predictions for the 50k regression dataset
# due to the interruption during the initial run.

# Define paths
preds_dir = Path("test_results/Regression/predictions")
scores_dir = Path("test_results/Regression/scores")

partial_preds_path = preds_dir / "test_preds_50k_tabpfn.parquet"
test_scores_path = scores_dir / "test_scores_50k_tabpfn.csv"

# Load already computed partial test predictions and previous scores
completed_preds_df = pd.read_parquet(partial_preds_path)
previous_scores_df = pd.read_csv(test_scores_path)
previous_pred_time = previous_scores_df["Pred Time (s)"].iloc[0]

# Ensure that the indices of the completed predictions are properly
# aligned with the test dataset
completed_indices = completed_preds_df.index
missing_indices = test_full_X.index.difference(completed_indices)

# Report on the status of predictions
total_count = len(test_full_X)
done_count = len(completed_indices)
missing_count = len(missing_indices)
print(f"Total test records: {total_count}")
print(f"Already predicted: {done_count}")
print(f"Missing records to predict: {missing_count}")
print(f"Previous accumulated Pred Time (s): {previous_pred_time:.2f}")

if missing_count > 0:
    # Slice the dataset strictly via physical unpredicted indices
    remaining_test_X = test_full_X.loc[missing_indices]
    remaining_test_y = test_full_y_reg.loc[missing_indices]

    # Initialize and fit the model quickly
    print("\nPreparing and fitting the model...")
    model = TabPFNRegressor.create_default_for_version(
        ModelVersion.V3,
        enhanced_fit_mode=True,
        enhanced_fit_mode_metric="rmse",
        enhanced_fit_mode_time_limit_s=2400,
        random_state=42,
    )

    start_time_fit = time.time()
    model.fit(train_50k_X, train_50k_y_reg)
    fit_time = time.time() - start_time_fit
    print(f"Fit completed in {fit_time:.2f} seconds.")

    # Predict the remaining data in batches
    batch_limit = 18000
    max_batch_size = min(200000 - len(train_50k_X), batch_limit)
    print(
        f"\nPredicting remaining {missing_count} rows in batches of {max_batch_size}..."
    )

    new_test_preds = []

    start_time_pred = time.time()
    for i in range(0, len(remaining_test_X), max_batch_size):
        batch_X = remaining_test_X.iloc[i : i + max_batch_size]
        print(f" -> Processing batch [{i} : {i + len(batch_X)}] ...")
        try:
            b_preds = model.predict(batch_X)
            new_test_preds.append(pd.Series(b_preds, index=batch_X.index))
        except Exception as e:
            print(f"\n[WARNING] Interrupted during Test data prediction.")
            print(f"Exact error: {e}")
            print("Saving partially recovered results so far.")
            break

    # Process newly acquired predictions
    if new_test_preds:
        new_preds_series = pd.concat(new_test_preds)
        new_pred_time = time.time() - start_time_pred

        # Calculate total prediction time
        total_pred_time = previous_pred_time + new_pred_time

        # Match indices for the successfully predicted subset using strict physical index
        successfully_predicted_y = remaining_test_y.loc[new_preds_series.index]

        new_preds_df = pd.DataFrame(index=new_preds_series.index)
        new_preds_df["true_y"] = successfully_predicted_y
        new_preds_df["TabPFN"] = new_preds_series

        # Merge old and new predictions
        full_test_preds_df = pd.concat([completed_preds_df, new_preds_df])

        y_true_all = full_test_preds_df["true_y"]
        y_pred_all = full_test_preds_df["TabPFN"]

        # Evaluate full metrics
        test_rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
        test_r2 = r2_score(y_true_all, y_pred_all)
        test_mae = mean_absolute_error(y_true_all, y_pred_all)
        test_mape = mean_absolute_percentage_error(y_true_all, y_pred_all)

        print("\n--- COMBINED TEST EVALUATION ---")
        print(
            f"Predicted total {len(full_test_preds_df)} out of {total_count} records."
        )
        print(f"Total Accumulated Pred Time: {total_pred_time:.2f} s")
        print(
            f"TEST - RMSE: {test_rmse:.4f} | R2: {test_r2:.4f} | MAE: {test_mae:.4f} | MAPE: {test_mape:.4f}"
        )

        # Save combined predictions
        full_test_preds_df.to_parquet(partial_preds_path)
        print(f"\nSuccessfully saved combined predictions to: {partial_preds_path}")

        # Update scores
        df_scores_test = pd.DataFrame(
            [
                {
                    "Dataset": "50k",
                    "Model": "TabPFN",
                    "RMSE": test_rmse,
                    "R2": test_r2,
                    "MAE": test_mae,
                    "MAPE": test_mape,
                    "Fit Time (s)": 3012,
                    "Pred Time (s)": total_pred_time,
                }
            ]
        )
        df_scores_test.to_csv(test_scores_path, index=False)
        print(f"Successfully saved updated scores to: {test_scores_path}")

    else:
        print("No new test data batches were processed (failed immediately).")
else:
    print("All records have already been predicted. Nothing to do!")

Total test records: 268608
Already predicted: 200000
Missing records to predict: 68608
Previous accumulated Pred Time (s): 5655.50

Preparing and fitting the model...
00:00 Fitting... -

The provided train set hashes match previously uploaded train sets.


00:01 Fitting... Done!
Fit completed in 1.73 seconds.

Predicting remaining 68608 rows in batches of 18000...
 -> Processing batch [0 : 18000] ...
04:46 Predicting... Done!
 -> Processing batch [18000 : 36000] ...
04:43 Predicting... Done!
 -> Processing batch [36000 : 54000] ...
04:44 Predicting... Done!
 -> Processing batch [54000 : 68608] ...
04:37 Predicting... Done!

--- COMBINED TEST EVALUATION ---
Predicted total 268608 out of 268608 records.
Total Accumulated Pred Time: 6788.90 s
TEST - RMSE: 0.3551 | R2: 0.0511 | MAE: 0.2427 | MAPE: 56270932594.4490

Successfully saved combined predictions to: test_results/Regression/predictions/test_preds_50k_tabpfn.parquet
Successfully saved updated scores to: test_results/Regression/scores/test_scores_50k_tabpfn.csv


In [ ]:
# Fit the models again due to used hashed data in original run

# Shuffle to not use hashed data in the same order as before
shuffled_df = train_df.tail(50000).sample(frac=1, random_state=42)
train_50k_X_shuffled = shuffled_df.drop(["target", "target_annual_roi"], axis=1)
train_50k_y_reg_shuffled = shuffled_df["target_annual_roi"]

In [ ]:
# Fit
model = TabPFNRegressor.create_default_for_version(
                ModelVersion.V3,
                enhanced_fit_mode=True,
                enhanced_fit_mode_metric="rmse",
                enhanced_fit_mode_time_limit_s=2400,
                random_state=42
            )
model.fit(train_50k_X_shuffled, train_50k_y_reg_shuffled)

41:10 Fitting... Done!


TabPFNRegressor(enhanced_fit_mode=True, enhanced_fit_mode_metric='rmse',
                enhanced_fit_mode_time_limit_s=2400, model_path='v3_default',
                random_state=42)

In [ ]:
# Same for 100k Classification
shuffled_df = train_df.tail(100000).sample(frac=1, random_state=43)

train_100k_X_shuffled = shuffled_df.drop(["target", "target_annual_roi"], axis=1)
train_100k_y_cat_shuffled = shuffled_df["target"]

In [ ]:
# Fit
model = TabPFNClassifier.create_default_for_version(
                ModelVersion.V3,
                enhanced_fit_mode=True,
                enhanced_fit_mode_metric="roc_auc",
                enhanced_fit_mode_time_limit_s=2400,
                random_state=42
            )
model.fit(train_100k_X_shuffled, train_100k_y_cat_shuffled)

18:43 Fitting... Done!


TabPFNClassifier(enhanced_fit_mode=True, enhanced_fit_mode_metric='roc_auc',
                 enhanced_fit_mode_time_limit_s=2400, model_path='v3_default',
                 random_state=42)